# Lab 5



Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


In [10]:
# Question 1

# Lay out a matrix class
class matrix:
    def __init__(self, a, b=None):
        # method assuming n and m are passed
        if b is not None:
            n = a
            m = b

            # checking for errors
            if not(isinstance(n,int) and isinstance(m,int)):
                raise TypeError("matrix(n,m): n and m must be integer values")
            if n <= 0 or m <= 0:
                raise ValueError("matrix(n,m): n and m must be positive values")

            # conversion to the nested list style, easier to work with and follows the flow of Lab 4
            self.data = []
            for i in range(n):
                row = []
                for j in range(m):
                    row.append(0.0)
                self.data.append(row)
            return

        # other method, B, assuming the matrix is a list of lists already
        L = a

        # must be a list
        if not isinstance(L, list) or len(L) == 0:
            raise ValueError("Must be list of lists")

        # every row must be a list
        for row in L:
            if not isinstance(row, list):
                raise ValueError("Each row must be a list")

        # all rows must have SAME length (rectangular matrix)
        row_len = len(L[0])
        for row in L:
            if len(row) != row_len:
                raise ValueError("Rows must have same length")

        # copy values and convert to floats
        self.data = []
        for row in L:
            new_row = []
            for value in row:
                new_row.append(float(value))
            self.data.append(new_row)

    # this controls what is printed when displaying the matrix, using this so it doesnt just print a mem address
    def __repr__(self):
        return str(self.data)

    # second bullet point, allows M[i][j] and M[i,j], __getitem__ from lecture is most appropriate
    def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key
            return self.data[i][j]

        return self.data[key]

    # This __setitem__ function wasnt from lecture, but it is the most applicable in this scenario
    # literal M1 = M2 will not suffice
    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = float(value)
            return

        if isinstance(key, slice):

            # determine source data
            if isinstance(value, matrix):
                src = value.data
            else:
                src = value

            # validate rectangular structure
            if not isinstance(src, list) or len(src) == 0:
                raise ValueError("Invalid source")

            row_len = len(src[0])
            for row in src:
                if len(row) != row_len:
                    raise ValueError("Invalid source")

            # ensure same size
            if len(src) != len(self.data) or row_len != len(self.data[0]):
                raise ValueError("Size mismatch")

            # copy values element-by-element
            for i in range(len(self.data)):
                for j in range(len(self.data[0])):
                    self.data[i][j] = float(src[i][j])
            return

In [12]:
# Q1 test

print("---Testing for question 1---")
print()

print("Zero matrix")
M = matrix(2,2)
print(M)

print("\nMatrix from list")
A = matrix([[1,2],[3,4]])
print(A)

print("\nTry ragged matrix")
try:
    matrix([[1,2],[3]])
except:
    print("Error raised")

print("\nIndex with M[i][j]")
A[0][0] = 9
print(A)

print("\nIndex with M[i,j]")
A[1,1] = 8
print(A)

print("\nAssignment from another matrix")
B = matrix(2,2)
B[:] = A
print(B)

print("\nAssignment from list")
B[:] = [[5,5],[6,6]]
print(B)

print("\nTry wrong size assignment")
try:
    B[:] = [[1,2,3],[4,5,6]]
except:
    print("Error raised")

---Testing for question 1---

Zero matrix
[[0.0, 0.0], [0.0, 0.0]]

Matrix from list
[[1.0, 2.0], [3.0, 4.0]]

Try ragged matrix
Error raised

Index with M[i][j]
[[9, 2.0], [3.0, 4.0]]

Index with M[i,j]
[[9, 2.0], [3.0, 8.0]]

Assignment from another matrix
[[9.0, 2.0], [3.0, 8.0]]

Assignment from list
[[5.0, 5.0], [6.0, 6.0]]

Try wrong size assignment
Error raised


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * (Extra credit) Modify `__getitem__` implemented above to support slicing.
        

In [18]:
# Question 2

# Lay out a matrix class
class matrix:
    def __init__(self, a, b=None):
        # method assuming n and m are passed
        if b is not None:
            n = a
            m = b

            # checking for errors
            if not(isinstance(n,int) and isinstance(m,int)):
                raise TypeError("matrix(n,m): n and m must be integer values")
            if n <= 0 or m <= 0:
                raise ValueError("matrix(n,m): n and m must be positive values")

            # conversion to the nested list style, easier to work with and follows the flow of Lab 4
            self.data = []
            for i in range(n):
                row = []
                for j in range(m):
                    row.append(0.0)
                self.data.append(row)
            return

        # other method, B, assuming the matrix is a list of lists already
        L = a

        # must be a list
        if not isinstance(L, list) or len(L) == 0:
            raise ValueError("Must be list of lists")

        # every row must be a list
        for row in L:
            if not isinstance(row, list):
                raise ValueError("Each row must be a list")

        # all rows must have SAME length (rectangular matrix)
        row_len = len(L[0])
        for row in L:
            if len(row) != row_len:
                raise ValueError("Rows must have same length")

        # copy values and convert to floats
        self.data = []
        for row in L:
            new_row = []
            for value in row:
                new_row.append(float(value))
            self.data.append(new_row)

    # this controls what is printed when displaying the matrix, using this so it doesnt just print a mem address
    def __repr__(self):
        return str(self.data)

    # second bullet point, allows M[i][j] and M[i,j], __getitem__ from lecture is most appropriate
    def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key
            return self.data[i][j]

        return self.data[key]

    # This __setitem__ function wasnt from lecture, but it is the most applicable in this scenario
    # literal M1 = M2 will not suffice
    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = float(value)
            return

        if isinstance(key, slice):

            # determine source data
            if isinstance(value, matrix):
                src = value.data
            else:
                src = value

            # validate rectangular structure
            if not isinstance(src, list) or len(src) == 0:
                raise ValueError("Invalid source")

            row_len = len(src[0])
            for row in src:
                if len(row) != row_len:
                    raise ValueError("Invalid source")

            # ensure same size
            if len(src) != len(self.data) or row_len != len(self.data[0]):
                raise ValueError("Size mismatch")

            # copy values element-by-element
            for i in range(len(self.data)):
                for j in range(len(self.data[0])):
                    self.data[i][j] = float(src[i][j])
            return

    # the shape method, returns a tupple (n,m) of the shape of the matrix
    def shape(self):
        return(len(self.data), len(self.data[0]))

    # transpose method, this will return a new matrix instance which is the transpose of the matrix
    # in hindsight, zip can be used here from lecture, so code is now changed to work w/ zip
    def transpose(self):
        return matrix([list(col) for col in zip(*self.data)])

    #old transpose for reference
    #def transpose(self):
    #n = len(self.data)
    #m = len(self.data[0])
    #
    #T = []
    #for j in range(m):
    #    row = []
    #    for i in range(n):
    #        row.append(self.data[i][j])
    #    T.append(row)
    #
    #return matrix(T)

    # row method
    def row(self, n):
        if not isinstance(n,int):
            raise TypeError("row(n): n needs to be an integer value")
        if n < 0 or n >= len(self.data):
            raise IndexError("row(n): n is out of range")
        return matrix([self.data[n][:]])

    # column method
    def column(self, n):
        if not isinstance(n, int):
            raise TypeError("column(n): n must be an integer value")
        if n < 0 or n >= len(self.data[0]):
            raise IndexError("column(n): n is out of range")
        col = []
        for i in range(len(self.data)):
            col.append([self.data[i][n]])
        return matrix(col)

    def to_list(self):
        out = []
        for row in self.data:
            out.append(row[:])
        return out

    # ChatGPT implementation, placeholder for the time being due to previous repetitive error
    # def block(self, n_0, n_1, m_0, m_1):
    #     if not (isinstance(n_0, int) and isinstance(n_1, int) and isinstance(m_0, int) and isinstance(m_1, int)):
    #         raise TypeError("block(n_0,n_1,m_0,m_1): inputs must be integers")
    # 
    #     n, m = self.shape()
    # 
    #     if n_0 < 0 or n_1 > n or n_0 > n_1:
    #         raise IndexError("block: row bounds out of range")
    #     if m_0 < 0 or m_1 > m or m_0 > m_1:
    #         raise IndexError("block: column bounds out of range")
    #
    #     B = []
    #     for i in range(n_0, n_1):
    #         B.append(self.data[i][m_0:m_1])
    #     return matrix(B)

    # block(n_0,n_1,m_0,m_1) that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows.
    # Interpreting shape() = (n,m) as (rows, cols):

    # Similar in form to chatGPTs method, return just more polished but inherently the same functionality
    def block(self, n_0, n_1, m_0, m_1):
        n, m = self.shape()

        if n_0 < 0 or n_1 > n or n_0 > n_1:
            raise IndexError("block: row bounds are out of range")
        if m_0 < 0 or m_1 > m or m_0 > m_1:
            raise IndexError("block: column bounds are out of range")

        return matrix([row[m_0:m_1] for row in self.data[n_0:n_1]])

    # come back later for __getitem__ modification

In [19]:
print("--- Q2 bare bones tests ---")
print()

A = matrix([[1,2,3],[4,5,6]])
print("A =", A)

print("\nshape")
print(A.shape())

print("\ntranspose")
print(A.transpose())

print("\nrow(0)")
print(A.row(0))

print("\nrow(1)")
print(A.row(1))

print("\ncolumn(0)")
print(A.column(0))

print("\ncolumn(2)")
print(A.column(2))

print("\nto_list")
print(A.to_list())

print("\nblock(0,2,1,3)")
print(A.block(0,2,1,3))

print("\nblock(1,2,0,2)")
print(A.block(1,2,0,2))

--- Q2 bare bones tests ---

A = [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]

shape
(2, 3)

transpose
[[1.0, 4.0], [2.0, 5.0], [3.0, 6.0]]

row(0)
[[1.0, 2.0, 3.0]]

row(1)
[[4.0, 5.0, 6.0]]

column(0)
[[1.0], [4.0]]

column(2)
[[3.0], [6.0]]

to_list
[[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]

block(0,2,1,3)
[[2.0, 3.0], [5.0, 6.0]]

block(1,2,0,2)
[[4.0, 5.0]]


In [31]:
# This is the __getitem__ multiplication, didnt incorporate above, but will incorporate in the .py file used for exercise 6
# Below is just the implementation without execution/test

#    # second bullet point, allows M[i][j] and M[i,j], __getitem__ from lecture is most appropriate
#    def __getitem__(self, key):
#        if isinstance(key, tuple):
#            i, j = key
#            return self.data[i][j]
#
#        return self.data[key]

def __getitem__(self,key):
    # the first inscance is w two indices
    if isinstance(key, tuple):
        r, c = key

        # M [i,j]
        if isinstance(r,int) and isinstance(c, int):
            return self.data[r][c]

        # row selection
        if isinstance(r, int):
            rows = [self.data[r]]
        else:
            rows = self.data[r]
            # r is slice

        # column selection
        if isinstance(c, int):
            return matrix([[row[c]] for row in rows])
        else:
            return matrix([row[c] for row in rows])
            # this instace c is slice

        # M [i:j}
        if isinstance(key, slice):
            return matrix([row[:] for row in self.data[key]])

        # M[i]
        return self.data[key]

In [32]:
# chat gpt generated test of the above function, it looks like it worked so it will now go in the .py for import

print("\n==============================")
print("   MATRIX SLICING DEMONSTRATION")
print("==============================\n")

A = matrix([[1,2,3],
            [4,5,6],
            [7,8,9]])

print("We begin with the original 3x3 matrix A:")
print(A)

print("\n--------------------------------")
print("Accessing an entire row using a single index")
print("Selecting row 1 (second row of A)")
print("Expected: [4, 5, 6]")
print("Result:")
print(A[1])

print("\n--------------------------------")
print("Accessing a single element using nested indexing")
print("Row 1, Column 2")
print("Expected: 6")
print("Result:")
print(A[1][2])

print("\n--------------------------------")
print("Accessing the same element using tuple indexing")
print("A[1,2]")
print("Expected: 6")
print("Result:")
print(A[1,2])

print("\n--------------------------------")
print("Extracting the first two rows of A")
print("This forms a new matrix using row slicing A[0:2]")
print("Expected: [[1,2,3],[4,5,6]]")
print("Result:")
print(A[0:2])

print("\n--------------------------------")
print("Extracting an entire column using slice notation")
print("Selecting column 1 across all rows: A[:,1]")
print("Expected column vector with values 2,5,8")
print("Result:")
print(A[:,1])

print("\n--------------------------------")
print("Extracting a full row but keeping matrix shape")
print("Using A[1,:] to get row 1 as a matrix")
print("Expected: [[4,5,6]]")
print("Result:")
print(A[1,:])

print("\n--------------------------------")
print("Extracting a rectangular submatrix")
print("Rows 0 through 1 and columns 1 through 2")
print("A[0:2, 1:3]")
print("Expected: [[2,3],[5,6]]")
print("Result:")
print(A[0:2,1:3])

print("\n--------------------------------")
print("Extracting the last column as a column matrix")
print("All rows, column index 2")
print("A[0:3, 2]")
print("Expected: [[3],[6],[9]]")
print("Result:")
print(A[0:3,2])

print("\n--------------------------------")
print("Final confirmation: original matrix remains unchanged")
print(A)

print("\n===== END OF SLICING DEMO =====")


   MATRIX SLICING DEMONSTRATION

We begin with the original 3x3 matrix A:
[[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]

--------------------------------
Accessing an entire row using a single index
Selecting row 1 (second row of A)
Expected: [4, 5, 6]
Result:
[4.0, 5.0, 6.0]

--------------------------------
Accessing a single element using nested indexing
Row 1, Column 2
Expected: 6
Result:
6.0

--------------------------------
Accessing the same element using tuple indexing
A[1,2]
Expected: 6
Result:
6.0

--------------------------------
Extracting the first two rows of A
This forms a new matrix using row slicing A[0:2]
Expected: [[1,2,3],[4,5,6]]
Result:
[[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]

--------------------------------
Extracting an entire column using slice notation
Selecting column 1 across all rows: A[:,1]
Expected column vector with values 2,5,8
Result:
[4.0, 5.0, 6.0]

--------------------------------
Extracting a full row but keeping matrix shape
Using A[1,:] to get

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [21]:
# Question 3

def constant(n, m, c):
    # validation
    if not (isinstance(n, int) and isinstance(m, int)):
        raise TypeError("constant(n,m,c): n and m must be integer values")
    if n <= 0 or m <= 0:
        raise ValueError("constant(n,m,c): n and m must be positive values")

    X = []
    for i in range(n):
        row = []
        for j in range(m):
            # c is whatever is passed into constant, fills matrix with its floatr value
            row.append(float(c))
        X.append(row)
    return matrix(X)

# The following two just pass into the constant function with a defined c value
def zeros(n, m):
    return constant(n, m, 0.0)

def ones(n, m):
    return constant(n, m, 1.0)

# returns the identity matrix, so a diagonal of 1's
def eye(n):
    # validation
    if not isinstance(n, int):
        raise TypeError("eye(n): n must be of an integer value")
    if n <= 0:
        raise ValueError("eye(n): n must be a positive value")

    Y = []
    for i in range(n):
        row = []
        for j in range(n):
            if i == j:
                row.append(1.0)
            else:
                row.append(0.0)
        Y.append(row)
    return matrix(Y)

In [24]:
print("--- Q3 function tests ---")

print("\nconstant(2,3,7)")
C = constant(2,3,7)
print(C)

print("\nzeros(2,3)")
Z = zeros(2,3)
print(Z)

print("\nones(2,3)")
O = ones(2,3)
print(O)

print("\neye(4)")
I = eye(4)
print(I)

print("\nTry bad sizes")
try:
    zeros(-1, 3)
except:
    print("Error raised")

try:
    eye(0)
except:
    print("Error raised")

--- Q3 function tests ---

constant(2,3,7)
[[7.0, 7.0, 7.0], [7.0, 7.0, 7.0]]

zeros(2,3)
[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]

ones(2,3)
[[1.0, 1.0, 1.0], [1.0, 1.0, 1.0]]

eye(4)
[[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]]

Try bad sizes
Error raised
Error raised


4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

In [27]:
# Question 4

# Lay out a matrix class
class matrix:
    def __init__(self, a, b=None):
        # method assuming n and m are passed
        if b is not None:
            n = a
            m = b

            # checking for errors
            if not(isinstance(n,int) and isinstance(m,int)):
                raise TypeError("matrix(n,m): n and m must be integer values")
            if n <= 0 or m <= 0:
                raise ValueError("matrix(n,m): n and m must be positive values")

            # conversion to the nested list style, easier to work with and follows the flow of Lab 4
            self.data = []
            for i in range(n):
                row = []
                for j in range(m):
                    row.append(0.0)
                self.data.append(row)
            return

        # other method, B, assuming the matrix is a list of lists already
        L = a

        # must be a list
        if not isinstance(L, list) or len(L) == 0:
            raise ValueError("Must be list of lists")

        # every row must be a list
        for row in L:
            if not isinstance(row, list):
                raise ValueError("Each row must be a list")

        # all rows must have SAME length (rectangular matrix)
        row_len = len(L[0])
        for row in L:
            if len(row) != row_len:
                raise ValueError("Rows must have same length")

        # copy values and convert to floats
        self.data = []
        for row in L:
            new_row = []
            for value in row:
                new_row.append(float(value))
            self.data.append(new_row)

    # this controls what is printed when displaying the matrix, using this so it doesnt just print a mem address
    def __repr__(self):
        return str(self.data)

    # second bullet point, allows M[i][j] and M[i,j], __getitem__ from lecture is most appropriate
    def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key
            return self.data[i][j]

        return self.data[key]

    # This __setitem__ function wasnt from lecture, but it is the most applicable in this scenario
    # literal M1 = M2 will not suffice
    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = float(value)
            return

        if isinstance(key, slice):

            # determine source data
            if isinstance(value, matrix):
                src = value.data
            else:
                src = value

            # validate rectangular structure
            if not isinstance(src, list) or len(src) == 0:
                raise ValueError("Invalid source")

            row_len = len(src[0])
            for row in src:
                if len(row) != row_len:
                    raise ValueError("Invalid source")

            # ensure same size
            if len(src) != len(self.data) or row_len != len(self.data[0]):
                raise ValueError("Size mismatch")

            # copy values element-by-element
            for i in range(len(self.data)):
                for j in range(len(self.data[0])):
                    self.data[i][j] = float(src[i][j])
            return

    # the shape method, returns a tupple (n,m) of the shape of the matrix
    def shape(self):
        return(len(self.data), len(self.data[0]))

    # transpose method, this will return a new matrix instance which is the transpose of the matrix
    # in hindsight, zip can be used here from lecture, so code is now changed to work w/ zip
    def transpose(self):
        return matrix([list(col) for col in zip(*self.data)])

    #old transpose for reference
    #def transpose(self):
    #n = len(self.data)
    #m = len(self.data[0])
    #
    #T = []
    #for j in range(m):
    #    row = []
    #    for i in range(n):
    #        row.append(self.data[i][j])
    #    T.append(row)
    #
    #return matrix(T)

    # row method
    def row(self, n):
        if not isinstance(n,int):
            raise TypeError("row(n): n needs to be an integer value")
        if n < 0 or n >= len(self.data):
            raise IndexError("row(n): n is out of range")
        return matrix([self.data[n][:]])

    # column method
    def column(self, n):
        if not isinstance(n, int):
            raise TypeError("column(n): n must be an integer value")
        if n < 0 or n >= len(self.data[0]):
            raise IndexError("column(n): n is out of range")
        col = []
        for i in range(len(self.data)):
            col.append([self.data[i][n]])
        return matrix(col)

    def to_list(self):
        out = []
        for row in self.data:
            out.append(row[:])
        return out

    # ChatGPT implementation, placeholder for the time being due to previous repetitive error
    # def block(self, n_0, n_1, m_0, m_1):
    #     if not (isinstance(n_0, int) and isinstance(n_1, int) and isinstance(m_0, int) and isinstance(m_1, int)):
    #         raise TypeError("block(n_0,n_1,m_0,m_1): inputs must be integers")
    # 
    #     n, m = self.shape()
    # 
    #     if n_0 < 0 or n_1 > n or n_0 > n_1:
    #         raise IndexError("block: row bounds out of range")
    #     if m_0 < 0 or m_1 > m or m_0 > m_1:
    #         raise IndexError("block: column bounds out of range")
    #
    #     B = []
    #     for i in range(n_0, n_1):
    #         B.append(self.data[i][m_0:m_1])
    #     return matrix(B)

    # block(n_0,n_1,m_0,m_1) that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows.
    # Interpreting shape() = (n,m) as (rows, cols):

    # Similar in form to chatGPTs method, return just more polished but inherently the same functionality
    def block(self, n_0, n_1, m_0, m_1):
        n, m = self.shape()

        if n_0 < 0 or n_1 > n or n_0 > n_1:
            raise IndexError("block: row bounds are out of range")
        if m_0 < 0 or m_1 > m or m_0 > m_1:
            raise IndexError("block: column bounds are out of range")

        return matrix([row[m_0:m_1] for row in self.data[n_0:n_1]])

    # whats needed is scalar multiplication, add, subtract, matrix multiplication, element multiplication, and equals
    def scalarmul(self, c):
        out = []
        for row in self.data:
            new_row = []
            for value in row:
                # c is the scalar in cM
                new_row.append(float(value) * float(c))
            out.append(new_row)
        return matrix(out)

    # passing N into add func for M + N
    def add(self, N):
        # validation check to make sure they are the same size/compatible
        if not isinstance(N, matrix):
            raise TypeError("add(N): N must be a matrix insttance")
        if self.shape() != N.shape():
            raise ValueError("add(N): matrix size mismatch")

        out = []
        for i in range(len(self.data)):
            new_row = []
            for j in range(len(self.data)):
                # adding elements together of the same corresponding index
                new_row.append(self.data[i][j] + N.data[i][j])
            out.append(new_row)
        return matrix(out)

    def sub(self, N):
        # same validation check as always
        if not isinstance(N, matrix):
            raise TypeError("sub(N): N must be a matrix insttance")
        if self.shape() != N.shape():
            raise ValueError("sub(N): matrix size mismatch")

        # same process as last function, just subtracting instead
        out = []
        for i in range(len(self.data)):
            new_row = []
            for j in range(len(self.data[0])):
                new_row.append(self.data[i][j] - N.data[i][j])
            out.append(new_row)
        return matrix(out)

    # function for matrix multiplication, must follow traditional rules for matrix multiplication
    def mat_mult(self, N):
        if not isinstance(N, matrix):
            raise TypeError("mat_mult(N): N must be a matrix insttance")

        # typical matrix mult, m and n2 must equal one another for mult to take place
        # ex matrix 1x2 * 2x1 works because m and n2 align. n x m matrix <- format.
        n, m = self.shape()
        n2, m2 = N.shape()

        # other validation check for mult
        if m != n2:
            raise ValueError("mat_mult(N): size mismatch (inner dimensions must match)")

        # the mult process
        out = []
        # for loops stepping thru the matrix and multiplying respective positions
        for i in range(n):
            new_row = []
            for j in range(m2):
                s = 0.0
                for k in range(m):
                    s = s + self.data[i][k] * N.data[k][j]
                new_row.append(s)
            out.append(new_row)
        return matrix(out)

    def element_mult(self, N):
        if not isinstance(N, matrix):
            raise TypeError("element_mult(N): N must be a matrix insttance")
        if self.shape() != N.shape():
            raise ValueError("element_mult(N): matrix size mismatch")

        out = []
        for i in range(len(self.data)):
            new_row = []
            for j in range(len(self.data[0])):
                new_row.append(self.data[i][j] * N.data[i][j])
            out.append(new_row)
        return matrix(out)

    def equals(self, N):
        if not isinstance(N, matrix):
            return False
        if self.shape() != N.shape():
            return False

        for i in range(len(self.data)):
            for j in range(len(self.data[0])):
                if self.data[i][j] != N.data[i][j]:
                    return False

        return True

In [28]:
print("\n--- Q4 func tests ---")
print()

A = matrix([[1,2,3],[4,5,6]])
B = matrix([[10,20,30],[40,50,60]])
print("A =", A)
print("B =", B)

print("\nscalarmul")
print(A.scalarmul(2))

print("\nadd")
print(A.add(B))

print("\nsub")
print(B.sub(A))

print("\nelement_mult")
print(A.element_mult(B))

print("\nmat_mult")
X = matrix([[1,2],[3,4],[5,6]])   # 3x2
Y = matrix([[7,8,9],[10,11,12]])  # 2x3
print("X =", X)
print("Y =", Y)
print(X.mat_mult(Y))              # 3x3

print("\nequals")
print(A.equals(matrix([[1,2,3],[4,5,6]])))
print(A.equals(B))

print("\ntry add size mismatch")
try:
    A.add(matrix([[1,2],[3,4]]))
except:
    print("Error raised")

print("\ntry mat_mult size mismatch")
try:
    A.mat_mult(matrix([[1,2],[3,4]]))  # A is 2x3, this is 2x2 (inner mismatch)
except:
    print("Error raised")


--- Q4 func tests ---

A = [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]
B = [[10.0, 20.0, 30.0], [40.0, 50.0, 60.0]]

scalarmul
[[2.0, 4.0, 6.0], [8.0, 10.0, 12.0]]

add
[[11.0, 22.0], [44.0, 55.0]]

sub
[[9.0, 18.0, 27.0], [36.0, 45.0, 54.0]]

element_mult
[[10.0, 40.0, 90.0], [160.0, 250.0, 360.0]]

mat_mult
X = [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]
Y = [[7.0, 8.0, 9.0], [10.0, 11.0, 12.0]]
[[27.0, 30.0, 33.0], [61.0, 68.0, 75.0], [95.0, 106.0, 117.0]]

equals
True
False

try add size mismatch
Error raised

try mat_mult size mismatch
Error raised


5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


In [29]:
# Question 5

# Lay out a matrix class
class matrix:
    def __init__(self, a, b=None):
        # method assuming n and m are passed
        if b is not None:
            n = a
            m = b

            # checking for errors
            if not(isinstance(n,int) and isinstance(m,int)):
                raise TypeError("matrix(n,m): n and m must be integer values")
            if n <= 0 or m <= 0:
                raise ValueError("matrix(n,m): n and m must be positive values")

            # conversion to the nested list style, easier to work with and follows the flow of Lab 4
            self.data = []
            for i in range(n):
                row = []
                for j in range(m):
                    row.append(0.0)
                self.data.append(row)
            return

        # other method, B, assuming the matrix is a list of lists already
        L = a

        # must be a list
        if not isinstance(L, list) or len(L) == 0:
            raise ValueError("Must be list of lists")

        # every row must be a list
        for row in L:
            if not isinstance(row, list):
                raise ValueError("Each row must be a list")

        # all rows must have SAME length (rectangular matrix)
        row_len = len(L[0])
        for row in L:
            if len(row) != row_len:
                raise ValueError("Rows must have same length")

        # copy values and convert to floats
        self.data = []
        for row in L:
            new_row = []
            for value in row:
                new_row.append(float(value))
            self.data.append(new_row)

    # this controls what is printed when displaying the matrix, using this so it doesnt just print a mem address
    def __repr__(self):
        return str(self.data)

    # second bullet point, allows M[i][j] and M[i,j], __getitem__ from lecture is most appropriate
    def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key
            return self.data[i][j]

        return self.data[key]

    # This __setitem__ function wasnt from lecture, but it is the most applicable in this scenario
    # literal M1 = M2 will not suffice
    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = float(value)
            return

        if isinstance(key, slice):

            # determine source data
            if isinstance(value, matrix):
                src = value.data
            else:
                src = value

            # validate rectangular structure
            if not isinstance(src, list) or len(src) == 0:
                raise ValueError("Invalid source")

            row_len = len(src[0])
            for row in src:
                if len(row) != row_len:
                    raise ValueError("Invalid source")

            # ensure same size
            if len(src) != len(self.data) or row_len != len(self.data[0]):
                raise ValueError("Size mismatch")

            # copy values element-by-element
            for i in range(len(self.data)):
                for j in range(len(self.data[0])):
                    self.data[i][j] = float(src[i][j])
            return

    # the shape method, returns a tupple (n,m) of the shape of the matrix
    def shape(self):
        return(len(self.data), len(self.data[0]))

    # transpose method, this will return a new matrix instance which is the transpose of the matrix
    # in hindsight, zip can be used here from lecture, so code is now changed to work w/ zip
    def transpose(self):
        return matrix([list(col) for col in zip(*self.data)])

    #old transpose for reference
    #def transpose(self):
    #n = len(self.data)
    #m = len(self.data[0])
    #
    #T = []
    #for j in range(m):
    #    row = []
    #    for i in range(n):
    #        row.append(self.data[i][j])
    #    T.append(row)
    #
    #return matrix(T)

    # row method
    def row(self, n):
        if not isinstance(n,int):
            raise TypeError("row(n): n needs to be an integer value")
        if n < 0 or n >= len(self.data):
            raise IndexError("row(n): n is out of range")
        return matrix([self.data[n][:]])

    # column method
    def column(self, n):
        if not isinstance(n, int):
            raise TypeError("column(n): n must be an integer value")
        if n < 0 or n >= len(self.data[0]):
            raise IndexError("column(n): n is out of range")
        col = []
        for i in range(len(self.data)):
            col.append([self.data[i][n]])
        return matrix(col)

    def to_list(self):
        out = []
        for row in self.data:
            out.append(row[:])
        return out

    # ChatGPT implementation, placeholder for the time being due to previous repetitive error
    # def block(self, n_0, n_1, m_0, m_1):
    #     if not (isinstance(n_0, int) and isinstance(n_1, int) and isinstance(m_0, int) and isinstance(m_1, int)):
    #         raise TypeError("block(n_0,n_1,m_0,m_1): inputs must be integers")
    # 
    #     n, m = self.shape()
    # 
    #     if n_0 < 0 or n_1 > n or n_0 > n_1:
    #         raise IndexError("block: row bounds out of range")
    #     if m_0 < 0 or m_1 > m or m_0 > m_1:
    #         raise IndexError("block: column bounds out of range")
    #
    #     B = []
    #     for i in range(n_0, n_1):
    #         B.append(self.data[i][m_0:m_1])
    #     return matrix(B)

    # block(n_0,n_1,m_0,m_1) that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows.
    # Interpreting shape() = (n,m) as (rows, cols):

    # Similar in form to chatGPTs method, return just more polished but inherently the same functionality
    def block(self, n_0, n_1, m_0, m_1):
        n, m = self.shape()

        if n_0 < 0 or n_1 > n or n_0 > n_1:
            raise IndexError("block: row bounds are out of range")
        if m_0 < 0 or m_1 > m or m_0 > m_1:
            raise IndexError("block: column bounds are out of range")

        return matrix([row[m_0:m_1] for row in self.data[n_0:n_1]])

    # whats needed is scalar multiplication, add, subtract, matrix multiplication, element multiplication, and equals
    def scalarmul(self, c):
        out = []
        for row in self.data:
            new_row = []
            for value in row:
                # c is the scalar in cM
                new_row.append(float(value) * float(c))
            out.append(new_row)
        return matrix(out)

    # passing N into add func for M + N
    def add(self, N):
        # validation check to make sure they are the same size/compatible
        if not isinstance(N, matrix):
            raise TypeError("add(N): N must be a matrix insttance")
        if self.shape() != N.shape():
            raise ValueError("add(N): matrix size mismatch")

        out = []
        for i in range(len(self.data)):
            new_row = []
            for j in range(len(self.data)):
                # adding elements together of the same corresponding index
                new_row.append(self.data[i][j] + N.data[i][j])
            out.append(new_row)
        return matrix(out)

    def sub(self, N):
        # same validation check as always
        if not isinstance(N, matrix):
            raise TypeError("sub(N): N must be a matrix insttance")
        if self.shape() != N.shape():
            raise ValueError("sub(N): matrix size mismatch")

        # same process as last function, just subtracting instead
        out = []
        for i in range(len(self.data)):
            new_row = []
            for j in range(len(self.data[0])):
                new_row.append(self.data[i][j] - N.data[i][j])
            out.append(new_row)
        return matrix(out)

    # function for matrix multiplication, must follow traditional rules for matrix multiplication
    def mat_mult(self, N):
        if not isinstance(N, matrix):
            raise TypeError("mat_mult(N): N must be a matrix insttance")

        # typical matrix mult, m and n2 must equal one another for mult to take place
        # ex matrix 1x2 * 2x1 works because m and n2 align. n x m matrix <- format.
        n, m = self.shape()
        n2, m2 = N.shape()

        # other validation check for mult
        if m != n2:
            raise ValueError("mat_mult(N): size mismatch (inner dimensions must match)")

        # the mult process
        out = []
        # for loops stepping thru the matrix and multiplying respective positions
        for i in range(n):
            new_row = []
            for j in range(m2):
                s = 0.0
                for k in range(m):
                    s = s + self.data[i][k] * N.data[k][j]
                new_row.append(s)
            out.append(new_row)
        return matrix(out)

    def element_mult(self, N):
        if not isinstance(N, matrix):
            raise TypeError("element_mult(N): N must be a matrix insttance")
        if self.shape() != N.shape():
            raise ValueError("element_mult(N): matrix size mismatch")

        out = []
        for i in range(len(self.data)):
            new_row = []
            for j in range(len(self.data[0])):
                new_row.append(self.data[i][j] * N.data[i][j])
            out.append(new_row)
        return matrix(out)

    def equals(self, N):
        if not isinstance(N, matrix):
            return False
        if self.shape() != N.shape():
            return False

        for i in range(len(self.data)):
            for j in range(len(self.data[0])):
                if self.data[i][j] != N.data[i][j]:
                    return False

        return True

    # Overloads

    # case with scalar multiplication where the matrix is to the right of the num
    def __rmul__(self, scalar):
        if isinstance(scalar, (int, float)):
            return self.scalarmul(scalar)
        return NotImplemented

    # case in which it is M * 2 or M * N, so either scalar mult or matrix mult
    def __mul__(self, operand):
        # in the instance of scalar mult
        if isinstance(operand, (int, float)):
            return self.scalarmul(operand)
        # in the instance of matrix multiplication
        if isinstance(operand, matrix):
            return self.mat_mult(operand)
        return NotImplemented

    # case for matrix addition
    def __add__(self, other_matrix):
        if isinstance(other_matrix, matrix):
            return self.add(other_matrix)
        return NotImplemented

    # case for matrix subtraction
    def __sub__(self, other_matrix):
        if isinstance(other_matrix, matrix):
            return self.sub(other_matrix)
        return NotImplemented

    # case for matrices equaling one another
    def __eq__(self, other_matrix):
        return self.equals(other_matrix)

In [30]:
print("--- Problem 5 operator tests ---")

M = matrix([[1,2],[3,4]])
N = matrix([[5,6],[7,8]])

print("M =", M)
print("N =", N)

print("\n2*M")
print(2 * M)

print("\nM*2")
print(M * 2)

print("\nM+N")
print(M + N)

print("\nM-N")
print(M - N)

print("\nM*N")
print(M * N)

print("\nM==N")
print(M == N)

print("\nM==copy of M")
print(M == matrix([[1,2],[3,4]]))

print("\nAssignment behavior")

A = matrix([[1,2],[3,4]])
B = matrix([[9,9],[9,9]])

print("A =", A)
print("B =", B)

print("\nA = B (aliasing)")
A = B
B[0,0] = 100
print("A =", A)
print("B =", B)

print("\nValue copy using slicing")
A = matrix([[1,2],[3,4]])
B = matrix([[9,9],[9,9]])
A[:] = B
B[0,0] = 100
print("A =", A)
print("B =", B)

--- Problem 5 operator tests ---
M = [[1.0, 2.0], [3.0, 4.0]]
N = [[5.0, 6.0], [7.0, 8.0]]

2*M
[[2.0, 4.0], [6.0, 8.0]]

M*2
[[2.0, 4.0], [6.0, 8.0]]

M+N
[[6.0, 8.0], [10.0, 12.0]]

M-N
[[-4.0, -4.0], [-4.0, -4.0]]

M*N
[[19.0, 22.0], [43.0, 50.0]]

M==N
False

M==copy of M
True

Assignment behavior
A = [[1.0, 2.0], [3.0, 4.0]]
B = [[9.0, 9.0], [9.0, 9.0]]

A = B (aliasing)
A = [[100.0, 9.0], [9.0, 9.0]]
B = [[100.0, 9.0], [9.0, 9.0]]

Value copy using slicing
A = [[9.0, 9.0], [9.0, 9.0]]
B = [[100.0, 9.0], [9.0, 9.0]]


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [55]:
# Question 6

# library created for convenience
import matrix_lib as M

print("Matrix Algebra Properties")
print("-------------------------\n")

# Example matrices we will be working with

print("Example matrices")
A = M.matrix([[1, 2], [3, 4]])
B = M.matrix([[5, 6], [7, 8]])
C = M.matrix([[2, 0], [1, 2]])
# I is the identity matrix, thisll just be a 2x2 I matrix
I = M.eye(2)

print("Matrix A =", A)
print("Matrix B =", B)
print("Matrix C =", C)
print("Identity I =", I)

# First property
# (AB)C = A(BC)

print("\n\n")
print("\nAssociativity of matrix multiplication")
print("(AB)C = A(BC)")
print("-------------------------\n")

# left side and right side
ls = (A * B) * C
rs = A * (B * C)

# test segment
print("(AB)C =", ls)
print("A(BC) =", rs)
print("Are they equivalent?", ls == rs)


# Second property
# A(B + C) = AB + AC

print("\n\n")
print("\nDistributive property")
print("A(B + C) = AB + AC")
print("-------------------------\n")

ls = A * (B + C)
rs = (A * B) + (A * C)

print("A(B + C) =", ls)
print("AB + AC =", rs)
print("Are they equivalent?", ls == rs)

# Third property
# AB =/= BA

print("\n\n")
print("\nMatrix multiplication not commutative")
print("AB =/= BA")
print("-------------------------\n")

AB = A * B
BA = B * A

print("AB =", AB)
print("BA =", BA)
print("Are they equivalent?", AB == BA)

# Fourth property
# AI = A

print("\n\n")
print("\nIdentity matrix property")
print("AI = A")
print("-------------------------\n")

AI = A * I

print("A =", A)
print("AI =", AI)
print("Are they equivalent?", AI == A)

Matrix Algebra Properties
-------------------------

Example matrices
Matrix A = [[1.0, 2.0], [3.0, 4.0]]
Matrix B = [[5.0, 6.0], [7.0, 8.0]]
Matrix C = [[2.0, 0.0], [1.0, 2.0]]
Identity I = [[1.0, 0.0], [0.0, 1.0]]




Associativity of matrix multiplication
(AB)C = A(BC)
-------------------------

(AB)C = [[60.0, 44.0], [136.0, 100.0]]
A(BC) = [[60.0, 44.0], [136.0, 100.0]]
Are they equivalent? True




Distributive property
A(B + C) = AB + AC
-------------------------

A(B + C) = [[23.0, 26.0], [53.0, 58.0]]
AB + AC = [[23.0, 26.0], [53.0, 58.0]]
Are they equivalent? True




Matrix multiplication not commutative
AB =/= BA
-------------------------

AB = [[19.0, 22.0], [43.0, 50.0]]
BA = [[23.0, 34.0], [31.0, 46.0]]
Are they equivalent? False




Identity matrix property
AI = A
-------------------------

A = [[1.0, 2.0], [3.0, 4.0]]
AI = [[1.0, 2.0], [3.0, 4.0]]
Are they equivalent? True
